# Chapter 3: How speech recognition decoders read audio

This notebook runs every piece of code in the chapter, from the saved results, in order. Read the chapter alongside it: [How speech recognition decoders read audio](https://books.sugiv.fyi/voice_applications/asr-decoders.html).


In [ ]:
# Run this cell first. In Google Colab it fetches the companion repo; anywhere else, run the notebook
# from a clone of it. Either way it moves into the book's folder and installs what the chapter needs.
import os
import pathlib
import sys

if "google.colab" in sys.modules and not pathlib.Path("/content/voices-of-the-future").exists():
    !git clone -q --depth 1 https://github.com/sugix/voices-of-the-future /content/voices-of-the-future
candidates = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path("/content/voices-of-the-future")]
root = next(p for p in candidates if (p / "voice" / "sim").is_dir())
os.chdir(root / "voice")
%pip install -q -r ../requirements/results.txt

## The problem: long audio, short text, no alignment

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(1042)

# A character vocabulary. Index 0 is the CTC "blank".
VOCAB = ["_"] + list("abcdefghijklmnopqrstuvwxyz0123456789 -")
IDX = {c: i for i, c in enumerate(VOCAB)}
BLANK = 0
FRAME_S = 0.08  # 80 ms per encoder frame

## CTC: one guess per frame, then clean up

In [ ]:
def ctc_collapse(path):
    """Merge consecutive repeats, then drop blanks."""
    out, prev = [], None
    for s in path:
        if s != prev and s != "_":
            out.append(s)
        prev = s
    return "".join(out)


for p in ["hh_i", "hhhiii", "bell", "bel_ll"]:
    print(f"{p!r:>10} -> {ctc_collapse(p)!r}")

In [ ]:
def fake_posteriors(path, confidence=6.0, noise=0.8):
    """Frame-by-frame probabilities that peak on the given path."""
    logits = rng.normal(0, noise, size=(len(path), len(VOCAB)))
    for t, s in enumerate(path):
        logits[t, IDX[s]] += confidence
    logits -= logits.max(axis=1, keepdims=True)
    p = np.exp(logits)
    return p / p.sum(axis=1, keepdims=True)


def greedy_ctc(probs):
    return ctc_collapse([VOCAB[i] for i in probs.argmax(axis=1)])


path = "__hh__ww___ ___11__00___44__22___"
probs = fake_posteriors(path)
print(len(probs), "frames ->", repr(greedy_ctc(probs)))

shown = ["_", "h", "w", " ", "1", "0", "4", "2"]
fig, ax = plt.subplots(figsize=(9, 2.6))
ax.imshow(probs[:, [IDX[c] for c in shown]].T, aspect="auto", cmap="magma")
ax.set_yticks(range(len(shown)), ["blank" if c == "_" else repr(c) for c in shown])
ax.set_xlabel("encoder frame (80 ms each)")
ax.set_title("CTC frame probabilities for 'hw 1042'")
plt.tight_layout()

### How CTC learns without alignments

In [ ]:
def ctc_log_prob(probs, label_ids):
    """log P(label | audio), summed over every alignment (forward algorithm)."""
    ext = [BLANK]
    for c in label_ids:
        ext += [c, BLANK]
    S, T = len(ext), len(probs)
    logp = np.log(probs + 1e-12)
    alpha = np.full((T, S), -np.inf)
    alpha[0, 0] = logp[0, ext[0]]
    alpha[0, 1] = logp[0, ext[1]]
    for t in range(1, T):
        for s in range(S):
            a = alpha[t - 1, s]
            if s >= 1:
                a = np.logaddexp(a, alpha[t - 1, s - 1])
            if s >= 2 and ext[s] != BLANK and ext[s] != ext[s - 2]:
                a = np.logaddexp(a, alpha[t - 1, s - 2])
            alpha[t, s] = a + logp[t, ext[s]]
    return np.logaddexp(alpha[-1, -1], alpha[-1, -2])


SMALL = ["_", "a", "b"]
small = rng.dirichlet(np.ones(3), size=5)  # 5 frames over {blank, a, b}
paths = [p for p in itertools.product(range(3), repeat=5)
         if ctc_collapse([SMALL[i] for i in p]) == "ab"]
brute = sum(np.prod(small[np.arange(5), p]) for p in paths)
print(f"{len(paths)} of {3**5} paths collapse to 'ab'")
print(f"brute force: {brute:.6f}   forward algorithm: {np.exp(ctc_log_prob(small, [1, 2])):.6f}")

## Fixing "HW-1042" with what the shop already knows

In [ ]:
noisy = fake_posteriors(path)
for t, s in enumerate(path):
    if s == "4":
        row = np.full(len(VOCAB), 0.004)
        row[IDX["1"]], row[IDX["4"]] = 0.50, 0.42
        noisy[t] = row / row.sum()

heard = greedy_ctc(noisy)
print("greedy:", repr(heard))

In [ ]:
open_tickets = ["hw 1042", "hw 1024", "hw 2041", "hw 1402"]
scores = {t: ctc_log_prob(noisy, [IDX[c] for c in t]) for t in open_tickets}
for t, s in sorted(scores.items(), key=lambda kv: -kv[1]):
    print(f"{t}: log P = {s:8.2f}")
print("best open ticket:", max(scores, key=scores.get))

## Transducers: let the decoder read its own output

In [ ]:
def rnnt_steps(token_frames, T):
    return len(token_frames) + T  # every token, plus one blank per frame


def tdt_steps(token_frames, T, max_dur=4):
    steps, t, i = 0, 0, 0
    while t < T:
        if i < len(token_frames) and token_frames[i] == t:
            i += 1  # emit a token at this frame
        nxt = token_frames[i] if i < len(token_frames) else T
        steps += 1
        t += min(max_dur, nxt - t)  # duration predicted with the decision
    return steps


seconds = np.arange(1, 31)
rows = []
for sec in seconds:
    T = int(sec / FRAME_S)
    U = int(sec * 3)  # about 3 subword tokens per second of speech
    frames = np.linspace(0, T - 1, U).astype(int)
    rows.append((1, rnnt_steps(frames, T), tdt_steps(frames, T), U + 1))
ctc, rnnt, tdt, llm = np.array(rows).T

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(seconds, rnnt, label="RNN-T (frames + tokens)")
ax.plot(seconds, tdt, label="TDT (skips up to 4 frames)")
ax.plot(seconds, llm, label="LLM decoder (one per token)")
ax.plot(seconds, ctc, label="CTC (one parallel pass)")
ax.set_xlabel("audio length (s)")
ax.set_ylabel("sequential decoder steps")
ax.legend(frameon=False)
plt.tight_layout()
print(f"10 s of audio: RNN-T {rnnt[9]}, TDT {tdt[9]}, LLM {llm[9]}, CTC {ctc[9]}")

## LLM decoders: a language model that listens

## What streaming really costs

In [ ]:
def worst_case_delay_ms(chunk_ms, lookahead_ms, compute_ms):
    return chunk_ms + lookahead_ms + compute_ms


for chunk, look, comp in [(80, 0, 15), (160, 80, 15), (480, 480, 20), (1000, 1000, 25)]:
    print(f"chunk {chunk:4d} ms + look-ahead {look:4d} ms + compute {comp:2d} ms "
          f"= {worst_case_delay_ms(chunk, look, comp):5d} ms")

## Bonus: CTC tells you when each word was said

In [ ]:
def ctc_align(probs, label_ids):
    """Most likely alignment (Viterbi). Returns (first, last) frame per label."""
    ext = [BLANK]
    for c in label_ids:
        ext += [c, BLANK]
    S, T = len(ext), len(probs)
    logp = np.log(probs + 1e-12)
    score = np.full((T, S), -np.inf)
    back = np.zeros((T, S), dtype=int)
    score[0, 0], score[0, 1] = logp[0, ext[0]], logp[0, ext[1]]
    for t in range(1, T):
        for s in range(S):
            cands = [(score[t - 1, s], s)]
            if s >= 1:
                cands.append((score[t - 1, s - 1], s - 1))
            if s >= 2 and ext[s] != BLANK and ext[s] != ext[s - 2]:
                cands.append((score[t - 1, s - 2], s - 2))
            best, arg = max(cands)
            score[t, s], back[t, s] = best + logp[t, ext[s]], arg
    s = S - 1 if score[-1, S - 1] >= score[-1, S - 2] else S - 2
    states = [0] * T
    for t in range(T - 1, -1, -1):
        states[t], s = s, back[t, s]
    spans = {}
    for t, s in enumerate(states):
        if s % 2 == 1:
            k = s // 2
            a, b = spans.get(k, (t, t))
            spans[k] = (min(a, t), max(b, t))
    return [spans[k] for k in range(len(label_ids))]


text = "hw 1042"
spans = ctc_align(probs, [IDX[c] for c in text])
words, start = [], 0
for i, c in enumerate(text + " "):
    if c == " ":
        first, last = spans[start][0], spans[i - 1][1]
        words.append((text[start:i], first * FRAME_S, (last + 1) * FRAME_S))
        start = i + 1
for w, a, b in words:
    print(f"{w:>5}: {a:.2f} s to {b:.2f} s")

## On real models

In [ ]:
d = np.load("data/ch3/parakeet_ctc_c01.npz")
lp, vocab = d["logprobs"].astype(np.float32), list(d["vocab"])
blank = len(vocab) - 1
best = lp.argmax(axis=1)

tokens, prev, emitted = [], None, []
for t, i in enumerate(best):
    if i != prev and i != blank:
        tokens.append(vocab[i])
        emitted.append((t, vocab[i]))
    prev = i
print(f"{len(lp)} frames, {np.mean(best == blank):.0%} of them blank")
print("".join(tokens).replace("▁", " ").strip())

p = np.exp(lp)
times = np.arange(len(p)) * FRAME_S
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.plot(times, p[:, blank], label="blank", color="0.6")
ax.plot(times, np.delete(p, blank, axis=1).max(axis=1), label="best token")
for t, tok in emitted:
    ax.annotate(tok.replace("▁", ""), (t * FRAME_S, 1.02), ha="center", fontsize=7, rotation=60)
ax.set_ylim(0, 1.25)
ax.set_xlabel("seconds")
ax.set_ylabel("probability")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()

## Rerun the models

Everything above used the saved results. The cells below rerun the models that produced them, on a GPU (in Colab:
Runtime, then Change runtime type, then T4 GPU). Each step builds the software it needs in its own environment the
first time, which takes a few minutes, then runs. Set `VOTF_QUICK=2` first for a two-item check that everything
works; leave it unset for a full rerun, which overwrites the saved results. On GPUs without bfloat16, such as the
T4, the scripts use float16 automatically.

In [ ]:
import os
os.environ["VOTF_QUICK"] = "2"  # remove this line for a full rerun

Halfwheel's calls: Kokoro reads them, Chapter 2's phone line carries them.

In [ ]:
!bash tools/stack.sh tts
!~/.votf/envs/tts/bin/python sim/halfwheel_calls.py

In [ ]:
!bash tools/stack.sh nemo
!~/.votf/envs/nemo/bin/python sim/capture_ch3.py